## Discrete Self-Adaptation in Competitive Coevolutionary Algorithms Plots

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.ticker import FixedLocator
import numpy as np
import json
import os

### Plot Parameters

In [ ]:
FIGSIZE = (10, 6)
SUB_PLOT_FIGSIZE = (10,10)
COMBO_FIGSIZE = (9,13)

FONTSIZE = 14
plt.rcParams["font.size"] = FONTSIZE 

plot_dir_path = os.path.join("paper_runs", "plots")
os.makedirs(plot_dir_path, exist_ok=True)

label_map = {
    "payoff": "Payoff",
    "mutrate": "Mut. Threshold",
    "sdt": "SDT"
}

### Game and Run Parameters

In [ ]:
signed_vals = {"payoff"}

params = {
    "defendit": {
        "s1_fixed_stepsizes": {
            "overlay": ("payoff", 
                        ["AS_step_1", "AS_step_6", "AS_step_65", "AS_step_655"], 
                        SUB_PLOT_FIGSIZE, False, [], "Fig 3"),
            "experiment_mapping": {
                "AS_step_1": ("Step Size: 1", [28, 280, 2800]),
                "AS_step_6": ("Step Size: 6", [28, 280, 2800]),
                "AS_step_65": ("Step Size: 65", [28, 280, 2800]),
                "AS_step_655": ("Step Size: 655", [28, 280, 2800]),
                },
        },
        "s1_adaptive_and_static" : { 
            "combine_payoff_and_mutrate": (["SS", "AS", "AA"], "Fig 4", [1, 10, 100, 1_000]),
            "sdt": (["AA"], FIGSIZE, True, [], "Fig 5"),
            "experiment_mapping": {
                "SS": ('Static-Threshold Static-SDT (S-S)', [28, 280, 2800]),
                "AS": ('Adapt-Threshold Static-SDT (A-S)', [28, 280, 2800]),
                "AA": ('Adapt-Threshold Adapt-SDT (A-A)', [28, 280, 2800]), 
                },
        },
        "s1_vs_prev_paper": {
            "payoff": (["s1", "py_nonspatial"], SUB_PLOT_FIGSIZE, True, [], "Fig 6"),
            "experiment_mapping": {
                "s1": ("S1 Spatial + Discrete DefendIt-B Implementation", [28, 280, 2800]),
                "py_nonspatial": ("Python Non-Spatial + Continuous DefendIt-B Implementation", [28, 280, 2800]),
                },
        },
        "s1_budget_ordering" : {
            "combine_payoff_and_mutrate": (["inc_budget", "hill_budget", "dec_budget"], "Fig 7", [1, 10, 100, 1_000]),
            "experiment_mapping": {
                "inc_budget": ('Increasing Budget (28, 280, 2800)', [28, 280, 2800]),
                "hill_budget": ('Hill Budget (28, 2800, 28)', [28, 2800, 28]), 
                "dec_budget": ('Decreasing Budget (2800, 280, 28)', [2800, 280, 28]),
                },
        },
        "s1_long_run_2999_10_budgets": {
            "combine": ({
                "payoff": ([""], FIGSIZE, False, []),
                "mutrate": ([""], FIGSIZE, False, ["log", [1, 10, 100, 1_000, 10_000]]),
            }, "Fig 8"),
            "experiment_mapping": {"": ("", [3220, 1120, 2100, 4760, 1260, 3640, 2380, 560, 4340, 1680])},

        },
        "py_small_pop_more_gens": {
            "payoff": ([""], FIGSIZE, True, [], "Fig 10"),
            "mutrate": ([""], FIGSIZE, True, ["log"], "Fig 11"),
            "experiment_mapping": {"": ("", [28, 280, 2800])},
        },
        "py_vs_s1": {
            "payoff": (["s1", "py_spatial"], SUB_PLOT_FIGSIZE, False, [], "Fig 12"),
            "mutrate": (["s1", "py_spatial"], SUB_PLOT_FIGSIZE, False, ["log"], "Fig 13"),
            "sdt": (["s1", "py_spatial"], SUB_PLOT_FIGSIZE, False, [], "Fig 14"),
            "experiment_mapping": {
                "s1": ("S1 DefendIt-B Implementation", [28, 280, 2800]),
                "py_spatial": ("Python DefendIt-B Implementation", [28, 280, 2800]),
                },
        },
    },
    "abs_dif_game": {
        "dif_ordering": {
            "combine_payoff_and_mutrate": (["p25_p50_p75", "p25_p75_p25", "p75_p50_p25"], "Fig 9", [10, 1_000, 100_000]),
            "experiment_mapping": {
                "p25_p50_p75": ('25%, 50%, 75%', ["25%", "50%", "75%"]),
                "p25_p75_p25": ('25%, 75%, 25%', ["25%", "75%", "25%"]), 
                "p75_p50_p25": ('75%, 50%, 25%', ["75%", "50%", "25%"]),
            },
        }
    },
}

### Helper Functions:

In [ ]:
def cleanData(filepath, is_signed=True):
    """
    Loads 16-bit data from a CSV file, handling both signed and unsigned formats.
    Attempts to load data as uint16 first, falling back to int16. 
    If is_signed is True, returns a view of the data as int16.
    """
    try:
        data = np.loadtxt(filepath, delimiter=',', dtype=np.uint16)
    except:
        data = np.loadtxt(filepath, delimiter=',', dtype=np.int16)

    if is_signed:
        return data.view(np.int16)
    return data
    
def cleanDataFloat(filepath):
    """
    Loads data from a CSV file as 32-bit floating-point numbers.
    """
    return np.loadtxt(filepath, delimiter=',', dtype=np.float32)

def getAllRunsData(paths, is_signed=True):
    """
    Iterates through a list of file paths and loads data for each.
    Attempts to load files as 16-bit integers (using cleanData) first. 
    If that fails for any reason, falls back to loading all files as 32-bit floats.
    """
    try:
        return [cleanData(path, is_signed) for path in paths]
    except:
        return [cleanDataFloat(path) for path in paths]

In [ ]:
def forward(x):
    '''
    Maps data values (y) to screen space (0-1).
    '''
    return 1 - 2**(-2 * x)

def inverse(x):
    '''
    Maps screen space (0-1) back to data values.
    Clamping x to slightly less than 1 prevents log(0) or log(negative) errors
    '''
    x_clamped = np.minimum(x, 1 - 1e-9)
    return -0.5 * np.log2(1 - x_clamped)

In [ ]:
def make_graph(data_label, graph_params, data_vals):
    subplots, figsize, showx, special, figname = graph_params[data_label]

    output_file = os.path.join(plot_dir_path, f"{figname}.png")
    if os.path.exists(output_file):
        print(f"{output_file} already exists")
        return
    
    num_subplots = len(subplots)
    fig, axes = plt.subplots(num_subplots, 1, figsize=figsize) # TODO: sharex deal with 
    
    last_i = num_subplots - 1
    top_offset = last_i * -0.025
    add_legend = True
    is_signed = True if data_label in signed_vals else False

    if num_subplots == 1:
        axes = [axes]

    for i, (ax, exper_name) in enumerate(zip(axes, subplots)):
        title, settings = graph_params["experiment_mapping"][exper_name]
        run_params = data_vals[exper_name]['run_params']
        x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]

        att_runs = getAllRunsData(data_vals[exper_name][f'att_{data_label}_paths'], is_signed)
        def_runs = getAllRunsData(data_vals[exper_name][f'def_{data_label}_paths'], is_signed)

        if i == last_i:
            showx = True

        plot_on_ax(ax, x_axis, def_runs, att_runs, run_params, 
                label_map[data_label], title, settings, 
                alpha=0.3, add_legend=add_legend, top_offset=top_offset, showx=showx)
        add_legend = False

        if "log" in special:
            ax.set_yscale("log")

        if "old_paper_scale" in special:
            plt.yscale('function', functions=(forward, inverse))
            plt.ylim(0, 3)
            custom_ticks = [0, 0.5, 1, 1.5, 2, 2.5]
            plt.gca().yaxis.set_major_locator(FixedLocator(custom_ticks))
        elif "yticks" in special:
            # ax.set_yticks([1, 10, 100, 1000])
            ax.set_yticks([10, 1000, 100000])
            ax.tick_params(axis='y', which='minor', length=0)

    axes[-1].set_xlabel("Generation")
    plt.tight_layout()
    plt.savefig(output_file)
    plt.close(fig)

def plot_on_ax(ax, x_axis, 
               def_runs, att_runs, 
               run_params, y_label, title, budgets, 
               alpha=0.3, add_legend = True, top_offset = 0, showx = False):
    num_budget = len(budgets)

    att_run_medians = np.median(att_runs, axis=2)
    def_run_medians = np.median(def_runs, axis=2)

    att_mean_medians = np.mean(att_run_medians, axis=0)
    def_mean_medians = np.mean(def_run_medians, axis=0)

    att_median_std = np.std(att_run_medians, axis=0)
    def_median_std = np.std(def_run_medians, axis=0)
    
    ax.plot(x_axis, def_mean_medians, linewidth=2, label='Defender')
    ax.fill_between(x_axis, def_mean_medians - def_median_std, def_mean_medians + def_median_std, alpha=0.3)
    ax.plot(x_axis, att_mean_medians, linewidth=2, label='Attacker')
    ax.fill_between(x_axis, att_mean_medians - att_median_std, att_mean_medians + att_median_std, alpha=0.3)
    
    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        ax.axvline(x=budget_gens[i], color="black", linestyle='--', alpha=alpha)

    trans = ax.get_xaxis_transform()
    for i, bgen in enumerate(budget_gens):
        ax.text(bgen + budget_gens[1] // 2, 0.95+top_offset, f'{budgets[i]}', transform=trans, ha='center', va='bottom', fontsize=FONTSIZE, fontweight='bold')
    
    # ax.set_xlabel('Generation')
    ax.set_ylabel(f'Med. {y_label}')
    if not showx:
        ax.set_xticks([])

    ax.set_title(title)
    if add_legend:
        ax.legend()
    ax.grid(True, alpha=0.3)

def plot_with_bottom_lines(
    fig, gs1, gs2,
    x_axis, y_ticks,
    def_payoff_runs, att_payoff_runs,
    def_mutrate_runs, atk_mutrate_runs,
    run_params, y_label, sec_label,
    title, budgets, 
    alpha=0.3, top_offset=0,
    add_legend=True,
    showx=True
):
    # Top axis: your existing line plot
    ax_top = fig.add_subplot(gs1)
    plot_on_ax(
        ax_top,
        x_axis,
        def_payoff_runs,
        att_payoff_runs,
        run_params, y_label, title,
        budgets, 
        alpha=alpha, top_offset=top_offset,
        add_legend=add_legend,
        showx = showx
    )

    x_axis = np.asarray(x_axis)

    att_run_medians = np.median(atk_mutrate_runs, axis=2)
    def_run_medians = np.median(def_mutrate_runs, axis=2)

    att_mean_medians = np.mean(att_run_medians, axis=0)
    def_mean_medians = np.mean(def_run_medians, axis=0)

    # Bottom axis: bars, share x with top
    ax_bottom = fig.add_subplot(gs2, sharex=ax_top)
    ax_bottom.tick_params(axis='y', which='minor', length=0)
    ax_bottom.plot(x_axis, def_mean_medians, color='tab:blue',
                linewidth=2, label='Defender mutation rate')
    ax_bottom.plot(x_axis, att_mean_medians, color='tab:orange',
                linewidth=2, label='Attacker mutation rate')
    ax_bottom.set_ylabel(sec_label)
    if showx:
        ax_bottom.set_xlabel('Generation')
    else:
        ax_bottom.set_xticks([])
    ax_bottom.grid(True, alpha=0.3)
    ax_bottom.set_yscale('log')
    ax_bottom.set_yticks(y_ticks)

    num_budget = len(budgets)
    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        ax_bottom.axvline(x=budget_gens[i], color="black", linestyle='--')

    # Optional: hide x tick labels on top
    # plt.setp(ax_top.get_xticklabels(), visible=False)

def big_main_plot_with_small_subplot(graph_data, run_data):
    fig = plt.figure(figsize=COMBO_FIGSIZE)
    gs = fig.add_gridspec(8, 1, height_ratios=[3,0.6, 0.3, 3, 0.8, 0.3, 3, 0.8], hspace=0.04)

    showx = False
    add_legend = True

    subplots, figname, y_ticks = graph_data["combine_payoff_and_mutrate"]

    output_file = os.path.join(plot_dir_path, f"{figname}.png")
    if os.path.exists(output_file):
        print(f"{output_file} already exists")
        return
        
    num_subplots = len(subplots)
    last_i = num_subplots - 1
    top_offset = last_i * -0.025
    
    for i, exper_name in enumerate(subplots):
        title, settings = graph_data["experiment_mapping"][exper_name]
        run_params = run_data[exper_name]['run_params']
        x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]

        att_fit_runs = getAllRunsData(run_data[exper_name]['att_payoff_paths'])
        def_fit_runs = getAllRunsData(run_data[exper_name]['def_payoff_paths'])

        att_mutrate_runs = getAllRunsData(run_data[exper_name]['att_mutrate_paths'], False)
        def_mutrate_runs = getAllRunsData(run_data[exper_name]['def_mutrate_paths'], False)
    
        if i == last_i:
            showx = True

        plot_with_bottom_lines(fig, gs[3*i], gs[3*i + 1], x_axis, y_ticks,
                            def_fit_runs, att_fit_runs, 
                            def_mutrate_runs, att_mutrate_runs, 
                            run_params, "Payoff", "Med. Threshold", title, settings, 
                            alpha = 0.3, add_legend = add_legend, 
                            top_offset = top_offset, showx = showx)
        add_legend = False

    plt.savefig(output_file, bbox_inches="tight")
    plt.close(fig)

def combine_main_plots(graph_data, run_data):
    figname = graph_data["combine"][1]
    output_file = os.path.join(plot_dir_path, f"{figname}.png")
    if os.path.exists(output_file):
        print(f"{output_file} already exists")
        return

    num_subplots = sum([len(vals[0]) for vals in graph_data["combine"][0].values()])
    fig, axes = plt.subplots(num_subplots, 1, figsize=SUB_PLOT_FIGSIZE)

    if num_subplots == 1:
        axes = [axes]

    last_i = num_subplots - 1
    add_legend = True 
    top_offset = last_i * -0.025

    ax_i = 0

    for metric_key, vals in graph_data["combine"][0].items():
        is_signed = True if metric_key in signed_vals else False
        subplots, _, _, special = vals
        for exper_name in subplots:
            ax = axes[ax_i]
            title, settings = graph_data["experiment_mapping"][exper_name]
            current_run = run_data[exper_name]
            run_params = current_run['run_params']

            att_runs = getAllRunsData(current_run[f'att_{metric_key}_paths'], is_signed)
            def_runs = getAllRunsData(current_run[f'def_{metric_key}_paths'], is_signed)
            x_axis = list(range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"]))
            
            showx = (ax_i == last_i)

            plot_on_ax(ax, x_axis, def_runs, att_runs, run_params, 
                    label_map[metric_key], title, settings, 
                    alpha=0.3, add_legend=add_legend, 
                    top_offset=top_offset, showx=showx)

            for item in special:
                if "log" == item:
                    ax.set_yscale("log") 
                elif isinstance(item, list):
                    ax.set_yticks(item)
                    ax.tick_params(axis='y', which='minor', length=0)


            add_legend = False
            ax_i += 1
    plt.tight_layout()
    
    plt.savefig(output_file)
    plt.close(fig)

def overlay(graph_data, run_data):
    metric_key, subplot, figsize, _, _, figname = graph_data["overlay"]

    output_file = os.path.join(plot_dir_path, f"{figname}.png")
    if os.path.exists(output_file):
        print(f"{output_file} already exists")
        return

    fig, axs = plt.subplots(2, 1, figsize=figsize, sharex=True, sharey=False)
    budgets = next(iter(graph_data["experiment_mapping"].values()))[1]
    num_budget = len(budgets)
    top_offset = 2 * -0.025
    is_signed = True if metric_key in signed_vals else False

    run_params = run_data[next(iter(graph_data["experiment_mapping"].keys()))]["run_params"]
    x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]

    for exper_name in subplot:
        att_runs = getAllRunsData(run_data[exper_name][f'att_{metric_key}_paths'], is_signed)
        def_runs = getAllRunsData(run_data[exper_name][f'def_{metric_key}_paths'], is_signed)

        att_run_medians = np.median(att_runs, axis=2)
        def_run_medians = np.median(def_runs, axis=2)

        att_mean_medians = np.mean(att_run_medians, axis=0)
        def_mean_medians = np.mean(def_run_medians, axis=0)

        att_median_std = np.std(att_run_medians, axis=0)
        def_median_std = np.std(def_run_medians, axis=0)

        label = graph_data["experiment_mapping"][exper_name][0]
        axs[0].plot(x_axis, def_mean_medians, linewidth=2, label=label)
        axs[0].fill_between(x_axis, def_mean_medians - def_median_std, 
                            def_mean_medians + def_median_std, alpha=0.3, label='_nolegend_')
        
        axs[1].plot(x_axis, att_mean_medians, linewidth=2, label=label)
        axs[1].fill_between(x_axis, att_mean_medians - att_median_std, 
                            att_mean_medians + att_median_std, alpha=0.3, label='_nolegend_')

    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        axs[0].axvline(x=budget_gens[i], color="black", linestyle='--')
        axs[1].axvline(x=budget_gens[i], color="black", linestyle='--')

    trans0 = axs[0].get_xaxis_transform()
    trans1 = axs[1].get_xaxis_transform()
    for i, bgen in enumerate(budget_gens):
        axs[0].text(bgen + budget_gens[1] // 2, 0.95+top_offset, f'{budgets[i]}', transform=trans0, ha='center', va='bottom', fontsize=FONTSIZE, fontweight="bold")
        axs[1].text(bgen + budget_gens[1] // 2, 0.95+top_offset, f'{budgets[i]}', transform=trans1, ha='center', va='bottom', fontsize=FONTSIZE, fontweight="bold")

    for ax, ax_for in zip(axs, ["Defender", "Attacker"]):
        ax.set_ylabel('Avg. Med. Payoff')
        ax.grid(True, alpha=0.3)
        ax.set_title(f"Fixed Mutation Step Size: {ax_for}")
    axs[1].legend()
    axs[1].set_xlabel('Generation')

    plt.savefig(output_file, bbox_inches='tight')
    plt.close(fig)

### Sorting Data Files

In [ ]:
#--- Data Params ---
all_vals = {}
for game_name, runs in params.items():
    for run_name, graph_data in runs.items():
        filepath = os.path.join("paper_runs", game_name)
        full_run_path = os.path.join(filepath, run_name)

        all_vals.setdefault(game_name, {})[run_name] = {}
        cur_vals = all_vals[game_name][run_name]

        for exper_name in graph_data["experiment_mapping"]:
            exper_path = os.path.join(full_run_path, exper_name)
            cur_vals[exper_name] = {}
            
            all_paths = os.listdir(os.path.join(full_run_path, exper_name))
            cur_vals[exper_name]['att_payoff_paths'] = sorted([os.path.join(exper_path, x) for x in all_paths if "attacker_payoffs" in x])
            cur_vals[exper_name]['att_mutrate_paths'] = sorted([os.path.join(exper_path, x) for x in all_paths if "attacker_mutRates" in x])
            cur_vals[exper_name]['att_sdt_paths'] = sorted([os.path.join(exper_path, x) for x in all_paths if "attacker_sdts" in x])
            
            cur_vals[exper_name]['def_payoff_paths'] = sorted([os.path.join(exper_path, x) for x in all_paths if "defender_payoffs" in x])
            cur_vals[exper_name]['def_mutrate_paths'] = sorted([os.path.join(exper_path, x) for x in all_paths if "defender_mutRates" in x])
            cur_vals[exper_name]['def_sdt_paths'] = sorted([os.path.join(exper_path, x) for x in all_paths if "defender_sdts" in x])
            cur_vals[exper_name]['num_runs'] = len(cur_vals[exper_name]['att_payoff_paths'])

            try: # each exper has its own run_params.json in its subfolder
                with open(os.path.join(exper_path, "run_params.json"), "r") as f:
                    cur_vals[exper_name]['run_params'] = json.load(f)
            except: # shared run_params.json in outer folder
                with open(os.path.join(full_run_path, "run_params.json"), "r") as f:
                    cur_vals[exper_name]['run_params'] = json.load(f)

### Creating Paper Figures

In [ ]:
for game_name, runs in all_vals.items():
    for run_name, run_data in runs.items():
        graph_data = params[game_name][run_name]
        if "combine_payoff_and_mutrate" in graph_data:
            big_main_plot_with_small_subplot(graph_data, run_data)
        for metric_key in label_map:
            if metric_key in graph_data:
                make_graph(metric_key, graph_data, run_data)
        if "combine" in graph_data:
            combine_main_plots(graph_data, run_data)
        if "overlay" in graph_data:
            overlay(graph_data, run_data)